In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# إدارة الأداء: دالة Qiskit من Q-CTRL Fire Opal
*راجع [مرجع API](https://docs.quantum.ibm.com/api/functions/q-ctrl-performance-management)*

> **Note:** دوال Qiskit ميزة تجريبية متاحة فقط لمستخدمي خطط IBM Quantum&reg; Premium وFlex وOn-Prem (عبر IBM Quantum Platform API). هي في مرحلة إصدار معاينة وعرضة للتغيير.


<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.1
qiskit-ibm-runtime~=0.45.1
```
</AccordionItem>
</Accordion>
## نظرة عامة
يُسهّل Fire Opal Performance Management على أي شخص الحصول على نتائج ذات معنى من الحواسيب الكمومية على نطاق واسع دون الحاجة إلى أن يكون خبيرًا في الأجهزة الكمومية. عند تشغيل الدوائر باستخدام Fire Opal Performance Management، تُطبَّق تقنيات كبت الأخطاء المدفوعة بالذكاء الاصطناعي تلقائيًا، مما يُتيح توسيع نطاق المسائل الأكبر التي تشمل مزيدًا من البوابات والقبيتات. يُقلل هذا النهج من عدد اللقطات (shots) اللازمة للوصول إلى الإجابة الصحيحة دون أي تكلفة إضافية — مما يوفر وفرات كبيرة في وقت الحوسبة والتكلفة.

يكبت Performance Management الأخطاء ويزيد من احتمالية الحصول على الإجابة الصحيحة على الأجهزة الصاخبة. بعبارة أخرى، يزيد من نسبة الإشارة إلى الضوضاء. تُظهر الصورة التالية كيف أن الدقة المتزايدة التي يوفرها Performance Management يمكنها تقليل الحاجة إلى لقطات إضافية في حالة خوارزمية تحويل فورييه الكمومي (Quantum Fourier Transform) بعشرة قبيتات. بـ 30 لقطة فقط، يصل Q-CTRL إلى عتبة الثقة البالغة 99%، في حين يتطلب الإعداد الافتراضي (`QiskitRuntime` Sampler، `optimization_level`=3 و`resilience_level`=1، `ibm_sherbrooke`) 170,000 لقطة. بالحصول على الإجابة الصحيحة بشكل أسرع، توفر وقت حوسبة كبيرًا.

![تصور للأداء المُحسَّن في وقت التشغيل](../docs/images/guides/qctrl-performance-management/achieve_more.svg)

يمكن استخدام دالة Performance Management مع أي خوارزمية، ويمكنك استخدامها بسهولة بدلًا من [بدائيات Qiskit Runtime](/guides/primitives) القياسية. خلف الكواليس، تعمل تقنيات كبت أخطاء متعددة معًا لمنع الأخطاء من الحدوث أثناء التشغيل. جميع طرق pipeline في Fire Opal مُهيَّأة مسبقًا وغير مرتبطة بخوارزمية بعينها، مما يعني أنك دائمًا تحصل على أفضل أداء فور الاستخدام.

للحصول على إمكانية الوصول إلى Performance Management، [تواصل مع Q-CTRL](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com).
## الوصف
يوفر Fire Opal Performance Management خيارَين للتنفيذ مشابهَين لبدائيات Qiskit Runtime، بحيث يمكنك استبدال Sampler وEstimator من Q-CTRL بسهولة. سير العمل العام لاستخدام دالة Performance Management هو:
1. حدِّد دائرتك (والمؤثرات في حالة Estimator).
2. شغِّل الدائرة.
3. استرجع النتائج.

للحد من ضوضاء الأجهزة، يستخدم Fire Opal مجموعة من تقنيات كبت الأخطاء المدفوعة بالذكاء الاصطناعي الموضحة في الصورة التالية. مع Fire Opal، تكون جميع مراحل pipeline مؤتمتة بالكامل دون أي حاجة للتهيئة.

يُلغي pipeline الخاص بـ Fire Opal الحاجة إلى تكاليف إضافية، مثل زيادة وقت التشغيل الكمومي أو القبيتات الفيزيائية الإضافية. لاحظ أن وقت المعالجة الكلاسيكي لا يزال عاملًا مؤثرًا (راجع قسم [المعايير](#benchmarks) للاطلاع على التقديرات، حيث يعكس "الوقت الإجمالي" كلًا من المعالجة الكلاسيكية والكمومية). على عكس تخفيف الأخطاء الذي يستلزم تكلفة إضافية في شكل أخذ عينات، يعمل كبت أخطاء Fire Opal على مستوى البوابات والنبضات لمعالجة مصادر الضوضاء المختلفة ومنع احتمالية حدوث الأخطاء. بمنع الأخطاء، تنتفي الحاجة إلى المعالجة اللاحقة المكلفة.

تُصوِّر الصورة التالية طرق كبت الأخطاء التي يؤتمتها Fire Opal Performance Management.

![تصور لـ pipeline كبت الأخطاء](../docs/images/guides/qctrl-performance-management/error_suppression.svg)

تقدم الدالة بدائيَّتَين، Sampler وEstimator، ومدخلات ومخرجات كلتيهما تمتد لتشمل المواصفات المُنفَّذة لـ [بدائيات Qiskit Runtime V2](/guides/primitive-input-output#pubs).
## المعايير
تُظهر نتائج [معيارية الخوارزميات المنشورة](https://journals.aps.org/prapplied/abstract/10.1103/PhysRevApplied.20.024034) تحسينًا ملحوظًا في الأداء عبر خوارزميات مختلفة، تشمل Bernstein-Vazirani وتحويل فورييه الكمومي وبحث Grover وخوارزمية التحسين الكمومي التقريبي (QAOA) ومحلل القيم الذاتية المتغيري الكمومي (VQE). يقدم باقي هذا القسم مزيدًا من التفاصيل حول أنواع الخوارزميات التي يمكنك تشغيلها، فضلًا عن الأداء ووقت التشغيل المتوقعَين.

تُوضح الدراسات المستقلة التالية كيف يُمكِّن Performance Management من Q-CTRL البحث الخوارزمي على نطاق قياسي:
- [نوى الكم المُعامَلة وفعّالة الطاقة لتشخيص أعطال خدمات الشبكة](https://arxiv.org/abs/2405.09724v1) - تعلم نواة كمومية يصل إلى 50 قبيت
- [تقدير الفارق في الطور الكمومي المستند إلى الموتر لعرض توضيحي على نطاق واسع](https://arxiv.org/abs/2408.04946) - تقدير الطور الكمومي يصل إلى 33 قبيت
- [التعلم الهرمي للتعلم الآلي الكمومي: تقنية تدريب جديدة للدوائر الكمومية المتغيرة ذات النطاق الواسع](https://arxiv.org/abs/2311.12929) - تحميل بيانات كمومية يصل إلى 21 قبيت

يوفر الجدول التالي دليلًا تقريبيًا للدقة وأوقات التشغيل من عمليات المعيارية السابقة على `ibm_fez`. قد يختلف الأداء على الأجهزة الأخرى. وقت الاستخدام مبني على افتراض 10,000 لقطة لكل دائرة. "عدد القبيتات" المُشار إليه ليس حدًا صارمًا، بل يمثل عتبات تقريبية يمكنك عندها توقع دقة حل متسقة للغاية. وقد تم حل مسائل أكبر بنجاح، ويُشجَّع على الاختبار بعد تجاوز هذه الحدود.

| مثال | عدد القبيتات | الدقة | مقياس الدقة | الوقت الإجمالي (ثانية) | استخدام وقت التشغيل (ثانية) | البدائية (الوضع) |
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |------------- |
| Bernstein–Vazirani  |  50 قبيت    | 100%  | معدل النجاح (النسبة المئوية للتشغيلات التي تكون فيها الإجابة الصحيحة أعلى سلسلة بت عدًا)     | 10    | 8         | Sampler |
| تحويل فورييه الكمومي | 30 قبيت              | 100% | معدل النجاح (النسبة المئوية للتشغيلات التي تكون فيها الإجابة الصحيحة أعلى سلسلة بت عدًا)      | 10    | 8        | Sampler |
| تقدير الطور الكمومي  | 30 قبيت   | 99.9998%  | دقة الزاوية المُحددة: `1- abs(real_angle - angle_found)/pi`  | 10  | 8  | Sampler |
| محاكاة كمومية: نموذج Ising (15 خطوة) | 20 قبيت  | 99.775%   |  $A$ (المُعرَّف أدناه)  |  60 (لكل خطوة)  | 15 (لكل خطوة)   | Estimator |
| محاكاة كمومية 2: ديناميكيات جزيئية (20 نقطة زمنية) | 34 قبيت  |  96.78%  |  $A_{mean}$ (المُعرَّف أدناه)   | 10 (لكل نقطة زمنية)    | 6 (لكل نقطة زمنية)  | Estimator |

تعريف دقة قياس قيمة التوقع - المقياس $A$ مُعرَّف على النحو التالي:
$$
A = 1 - \frac{|\epsilon^{ideal} - \epsilon^{meas}|}{\epsilon^{ideal}_{max} - \epsilon^{ideal}_{min}},
$$
حيث $ \epsilon^{ideal} $ = قيمة التوقع المثالية، و$ \epsilon^{meas} $ = قيمة التوقع المقاسة، و$\epsilon^{ideal}_{max} $ = القيمة المثالية القصوى، و$\epsilon^{ideal}_{min}$ = القيمة المثالية الدنيا. و$A_{mean}$ هو ببساطة متوسط قيمة $A$ عبر قياسات متعددة.

يُستخدم هذا المقياس لأنه لا يتأثر بالإزاحات الكلية والقياس في نطاق القيم القابلة للتحقيق. بعبارة أخرى، بغض النظر عن رفع أو خفض نطاق قيم التوقع المحتملة أو زيادة انتشارها، يجب أن تظل قيمة $A$ ثابتة.
## البداية
يستخدم Fire Opal Performance Management الإصدار `2.0.0` من Qiskit، وهو الإصدار الموصى به. الإصدارات المدعومة هي Qiskit >=v`2.0.0`.
قم بالمصادقة باستخدام [مفتاح API لـ IBM Quantum Platform](http://quantum.cloud.ibm.com/)، واختر دالة Qiskit كما يلي. (يفترض هذا المقطع البرمجي أنك [حفظت حسابك](/guides/functions#install-qiskit-functions-catalog-client) بالفعل في بيئتك المحلية.)

In [1]:
# This cell is hidden from users. It hides the "...You have imported samplomatic..." warning.
import warnings

warnings.filterwarnings("ignore", message=".*You have imported samplomatic*")

In [2]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [3]:
# Access Function
perf_mgmt = catalog.load("q-ctrl/performance-management")

<Admonition type="note" title="Does this function support all IBM backends?">
If you want to use a backend that this function does not currently support, [reach out to Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com) to add support.
</Admonition>

## Estimator primitive

### Estimator example

Use Fire Opal Performance Management's Estimator primitive to determine the expectation value of a single circuit-observable pair.

In addition to the `qiskit-ibm-catalog` and `qiskit` packages, you will also use the `numpy` package to run this example. You can install this package by uncommenting the following cell if you are running this example in a notebook using the IPython kernel.

In [4]:
# %pip install numpy

**2. تشغيل الدائرة**

شغِّل الدائرة وحدِّد اختياريًا الـ backend وعدد اللقطات.

In [5]:
import numpy as np
from qiskit.circuit.library import iqp
from qiskit.quantum_info import random_hermitian, SparsePauliOp

n_qubits = 50

# Generate a random circuit
mat = np.real(random_hermitian(n_qubits, seed=1234))
circuit = iqp(mat)
circuit.measure_all()

# Define observables as a string
observable = SparsePauliOp("Z" * n_qubits)

In [6]:
# Create PUB tuple
estimator_pubs = [(circuit, observable)]

يمكنك استخدام [واجهات برمجة تطبيقات Qiskit Serverless](/guides/serverless) المألوفة للتحقق من حالة حمل عمل دالة Qiskit الخاصة بك:

In [7]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [8]:
# Run the circuit using Estimator
qctrl_estimator_job = perf_mgmt.run(
    primitive="estimator",
    pubs=estimator_pubs,
    backend_name=backend_name,
)

**3. استرجاع النتيجة**

In [9]:
qctrl_estimator_job.status()

'QUEUED'

النتائج لها نفس تنسيق [نتيجة Estimator](/guides/estimator-input-output):

In [10]:
# Retrieve the counts from the result list
result = qctrl_estimator_job.result()

The results have the same format as an [Estimator result](/docs/guides/estimator-input-output):

In [22]:
import numpy

result_str = str(result)

with numpy.printoptions(threshold=200):
    print(
        f"The result of the submitted job had {len(result)} PUB "
        f"and has a value:\n {result[0]}\n"
    )

print("The associated PubResult of this job has the following DataBins:")
print(f"{result[0].data}\n")

print(f"And this DataBin has attributes: {result[0].data.keys()}")

print("The expectation values measured from this PUB are:")
print(f"{result[0].data.evs}")

The result of the submitted job had 1 PUB
The result of the submitted job had 1 PUB and has a value:
 PubResult(data=DataBin(evs=0.0195, stds=0.9998098569228051), metadata={'precision': None})

The associated PubResult of this job has the following DataBins:
DataBin(evs=0.0195, stds=0.9998098569228051)

And this DataBin has attributes: dict_keys(['evs', 'stds'])
The expectation values measured from this PUB are:
0.0195


## Sampler primitive
### مثال على Sampler
استخدم Sampler primitive في Fire Opal Performance Management لتشغيل دائرة Bernstein–Vazirani. هذه الخوارزمية تُستخدم للعثور على سلسلة مخفية من مخرجات دالة صندوق أسود، وهي خوارزمية معيارية شائعة للاختبار لأن لها إجابة صحيحة واحدة فقط.

**1. إنشاء الدائرة**

حدِّد الإجابة الصحيحة للخوارزمية، وهي السلسلة الثنائية المخفية، ثم أنشئ دائرة Bernstein–Vazirani. يمكنك ضبط عرض الدائرة ببساطة عن طريق تغيير قيمة `circuit_width`.

In [12]:
import qiskit

circuit_width = 35
hidden_bitstring = "1" * circuit_width

# Create circuit, reserving one qubit for BV oracle
bv_circuit = qiskit.QuantumCircuit(circuit_width + 1, circuit_width)
bv_circuit.x(circuit_width)
bv_circuit.h(range(circuit_width + 1))
for input_qubit, bit in enumerate(reversed(hidden_bitstring)):
    if bit == "1":
        bv_circuit.cx(input_qubit, circuit_width)
bv_circuit.barrier()
bv_circuit.h(range(circuit_width + 1))
bv_circuit.barrier()
for input_qubit in range(circuit_width):
    bv_circuit.measure(input_qubit, input_qubit)

# Create PUB tuple
sampler_pubs = [(bv_circuit,)]

**2. تشغيل الدائرة**

شغِّل الدائرة وحدِّد اختياريًا Backend وعدد اللقطات (shots).

In [13]:
# Run the circuit using Sampler
qctrl_sampler_job = perf_mgmt.run(
    primitive="sampler",
    pubs=sampler_pubs,
    backend_name=backend_name,
)

تحقق من [حالة](/guides/functions#check-job-status) workload دالة Qiskit الخاصة بك أو استرجع [النتائج](/guides/functions#retrieve-results) كما يلي:

In [14]:
# Print the ID so you can use it later, if necessary
print(qctrl_sampler_job.job_id)

qctrl_sampler_job.status()

60fe2fa1-a860-43e4-8615-c6ac4180f93b


'QUEUED'

**3. Retrieve the result**

In [15]:
# Retrieve the job results
sampler_result = qctrl_sampler_job.result()

In [16]:
# Get results for the first (and only) PUB
pub_result = sampler_result[0]
counts = pub_result.data.c.get_counts()

print("Counts for the meas output register (limited to 30 results):")
for i, (bitstring, count) in enumerate(counts.items()):
    if i >= 50:
        print(f"  ... ({len(counts) - 30} more items)")
        break
    print(f"  {bitstring}: {count}")

Counts for the meas output register (limited to 30 results):
  11111111111111111111111111111111111: 1661
  11111111111111111111111111110111111: 60
  11111111111111111111111111111101111: 54
  11111111111111111111111111111110111: 54
  11111111111111011111111111111111111: 46
  11111111111111111110111111111111111: 44
  11111111111111111111111101111111111: 42
  11111111111111111111111110111111111: 42
  11111111111111110111111111111111111: 41
  11111111111111111111111111111111101: 39
  11111111111111111111101111111111111: 38
  11111111111111111111110111111111111: 38
  11111111111111111111111111101111111: 37
  11111111111111111111111111111111110: 36
  11111111111110111111111111111111111: 35
  11111111111111111111111111111011111: 32
  11111111111111101111111111111111111: 32
  01111111111111111111111111111111111: 27
  11111111111111111011111111111111111: 23
  11111111101111111111111111111111111: 22
  11111111111111111111111111111111011: 21
  11111111011111111111111111111111111: 20
  00000000000

**3. Plot the top bitstrings**

Plot the bitstring with the highest counts to see if the hidden bitstring was the mode.

In [17]:
import matplotlib.pyplot as plt


def plot_top_bitstrings(counts_dict, hidden_bitstring=None):
    # Sort and take the top 100 bitstrings
    top_100 = sorted(counts_dict.items(), key=lambda x: x[1], reverse=True)[
        :100
    ]
    if not top_100:
        print("No bitstrings found in the input dictionary.")
        return

    # Unzip the bitstrings and their counts
    bitstrings, counts = zip(*top_100)

    # Assign colors: purple if the bitstring matches hidden_bitstring,
    # otherwise gray
    colors = [
        "#680CE9" if bit == hidden_bitstring else "gray" for bit in bitstrings
    ]

    # Create the bar plot
    plt.figure(figsize=(15, 8))
    plt.bar(
        range(len(bitstrings)), counts, tick_label=bitstrings, color=colors
    )

    # Rotate the bitstrings for better readability
    plt.xticks(rotation=90, fontsize=8)
    plt.xlabel("Bitstrings")
    plt.ylabel("Counts")
    plt.title("Top 100 Bitstrings by Counts")

    # Show the plot
    plt.tight_layout()
    plt.show()

The hidden bitstring is highlighted in purple, and it should be the bitstring with the highest number of counts.

In [18]:
plot_top_bitstrings(counts, hidden_bitstring)

<Image src="../docs/images/guides/q-ctrl-performance-management/extracted-outputs/8106d906-0.avif" alt="Output of the previous code cell" />

**3. رسم أعلى السلاسل الثنائية**

ارسم السلسلة الثنائية ذات أعلى عدد تكرارات لترى إذا كانت السلسلة المخفية هي الأكثر شيوعًا.